# CAP 5610 Midterm — Task 2
## Neural Network Regression on the King County Housing Dataset

The goal of this task is to predict house prices using a neural network
regression model.

The target variable will be log-transformed before training. Numerical
predictors will be standardized, and Logistic sigmoid and ReLU activation
functions will be compared using the same network architecture.

In [1]:
# Numerical and tabular data
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

# Data preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Neural network regression
from sklearn.neural_network import MLPRegressor

# Model evaluation
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

## 1. Load the Training and Test Data

In [2]:
# Paths relative to the notebook directory.
train_path = "../data/king_county_train_.csv"
test_path = "../data/king_county_test_.csv"

# Load the supplied datasets.
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

Training shape: (16613, 19)
Test shape: (5000, 19)


In [3]:
print("Training columns:")
print(train_df.columns.tolist())

print("\nData types:")
print(train_df.dtypes)

print("\nFirst five rows:")
display(train_df.head())

print("\nMissing values in training data:")
missing_train = train_df.isna().sum()
print(missing_train[missing_train > 0])

print("\nMissing values in test data:")
missing_test = test_df.isna().sum()
print(missing_test[missing_test > 0])

Training columns:
['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']

Data types:
price              int64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
dtype: object

First five rows:


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,392000,3,1.50,1500,11975,1.0,0,0,4,7,1500,0,1970,0,98045,47.4658,-121.758,1510,13875
1,499950,3,2.50,3230,129578,1.0,0,0,4,8,2100,1130,1964,0,98014,47.6820,-121.913,2760,62059
2,510000,3,1.75,1480,7040,1.0,0,0,3,7,1480,0,1974,0,98033,47.6723,-122.169,2040,7810
3,648360,4,1.75,2260,7005,1.0,0,1,4,7,1130,1130,1947,0,98033,47.6895,-122.207,2330,9180
4,248000,4,2.25,2320,8760,1.0,0,0,4,8,1160,1160,1966,0,98032,47.3627,-122.285,1970,8690



Missing values in training data:
Series([], dtype: int64)

Missing values in test data:
Series([], dtype: int64)


## 3. Define the Target and Predictor Variables

`price` is the regression target.

Most predictors are numerical and will be standardized before being passed
to the neural network. `zipcode` is stored numerically in the dataset, but
it represents a geographic category rather than a continuous quantity, so
it is one-hot encoded instead of standardized.

Although the supplied data contain no missing values, imputation is kept
in the preprocessing pipeline so that missing values are handled
consistently if they occur.

In [4]:
# Separate predictors and target.
X = train_df.drop(columns=["price"])
y = train_df["price"]

# Keep the supplied test set separate for final evaluation.
X_test = test_df.drop(columns=["price"])
y_test = test_df["price"]

# Zip code is categorical even though it is stored as an integer.
categorical_features = ["zipcode"]

# All remaining predictors are treated as numerical.
numerical_features = [
    column for column in X.columns
    if column not in categorical_features
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long', 'sqft_living15', 'sqft_lot15']

Categorical features:
['zipcode']


## 4. Log-Transform the Target

House prices are strongly right-skewed. The target is therefore transformed
using

$$
y_{\log} = \log(1+y).
$$

The neural network is trained to predict log-price. Predictions will later
be transformed back to dollars using `np.expm1()` before MSE and $R^2$ are
calculated.

In [5]:
# Apply the required log transformation to the target.
y_log = np.log1p(y)

print("Original price range:")
print(f"${y.min():,.0f} to ${y.max():,.0f}")

print("\nLog-transformed target range:")
print(f"{y_log.min():.4f} to {y_log.max():.4f}")

Original price range:
$75,000 to $7,700,000

Log-transformed target range:
11.2253 to 15.8567


## 5. Create Training and Validation Sets

A validation subset is created from the provided training data so that
network configurations can be compared without using the supplied test set.
The test set is reserved for the final evaluation.

In [6]:
# Split the provided training data into training and validation subsets.
X_train, X_val, y_train_log, y_val_log = train_test_split(
    X,
    y_log,
    test_size=0.20,
    random_state=42
)

print("Training subset:", X_train.shape)
print("Validation subset:", X_val.shape)

Training subset: (13290, 18)
Validation subset: (3323, 18)
